### MergeFunders — merge one funder entity into another (one-off, parameterized)

Standard funder entity merge. First use: **F4320307874 "Wellcome" → F4320311904 "Wellcome Trust"**
(same ROR 029chgv08; Crossref registry holds two funder DOIs, 10.13039/100004440 and
10.13039/100010269, for the same organisation — year 1 report prep 2026-07-22).

**Mechanism — alias row, not delete.** Aggregator matching (Crossref / DataCite / EuropePMC)
resolves funders by DOI or name against `openalex.mid.funder`. Publishers keep depositing the
loser's funder DOI forever, so the loser row must SURVIVE as an alias: it keeps its `doi`
(and name) but gets `merge_into_id` set to the winner. Matchers resolve
`COALESCE(merge_into_id, funder_id)`; `CreateFundersAPI` filters `merge_into_id IS NULL`.
Deleting the row instead would silently drop every future work-funder link deposited under
the loser DOI (~26K works for Wellcome).

**What this notebook does (in order):**
1. Preflight: both rows exist, loser has no direct-ingest awards (aggregator-only allowlist).
2. Schema: add `merge_into_id` / `merge_into_date` to `mid.funder` (+ `common.funder` if present).
3. Winner absorbs loser display_name + alternate_titles into `alternate_titles` (name matchers).
4. Tombstone loser (`merge_into_id` = winner).
5. One-time remap of stored edges: `mid.work_funder`, `crossref_work_funders`,
   `datacite_work_funders`, `europepmc_work_funders`, `fulltext_work_funders`.
6. `openalex_awards_raw`: recompute award ids under the winner (`id = hash(funder_id:award)`
   is the cross-provenance dedup key), delete rows that collide with existing winner awards.
7. Elasticsearch: write the `merge-funders` redirect mapping; delete the loser doc from
   the funders index (the sync's delete-stale block also covers this).

**Ordering contract:** run this notebook BEFORE merging the `funder-merge-*` branch to main —
the branch's matcher/API-build edits reference `merge_into_id`, which this notebook creates.
After both: the scheduled chain (CreateAwards 3am → CreateFundersAPI → ES 5am, and the
end2end works refresh) converges works/awards onto the winner. See
`plans/funders/funder-merge-runbook.md`.

In [ ]:
%pip install elasticsearch==8.19.0

In [ ]:
dbutils.widgets.text("merge_from_id", "4320307874")   # loser (bare numeric, no F)
dbutils.widgets.text("merge_into_id", "4320311904")   # winner

MERGE_FROM = int(dbutils.widgets.get("merge_from_id"))
MERGE_INTO = int(dbutils.widgets.get("merge_into_id"))
assert MERGE_FROM != MERGE_INTO, "merge_from and merge_into must differ"
print(f"merging F{MERGE_FROM} -> F{MERGE_INTO}")

In [ ]:
# Preflight ---------------------------------------------------------------
rows = spark.sql(f"""
  SELECT funder_id, display_name, ror_id, doi, crossref_id, alternate_titles
  FROM openalex.mid.funder WHERE funder_id IN ({MERGE_FROM}, {MERGE_INTO})
""").collect()
by_id = {r.funder_id: r for r in rows}
assert MERGE_FROM in by_id, f"loser F{MERGE_FROM} not found in mid.funder"
assert MERGE_INTO in by_id, f"winner F{MERGE_INTO} not found in mid.funder"
for r in rows:
    print(r.funder_id, "|", r.display_name, "|", r.ror_id, "|", r.doi)

mid_cols = [f.name for f in spark.table("openalex.mid.funder").schema]
if "merge_into_id" in mid_cols:
    already = spark.sql(f"""
      SELECT funder_id, merge_into_id FROM openalex.mid.funder
      WHERE funder_id IN ({MERGE_FROM}, {MERGE_INTO}) AND merge_into_id IS NOT NULL
    """).collect()
    assert not already, f"row already merged: {already}"

if by_id[MERGE_FROM].ror_id != by_id[MERGE_INTO].ror_id:
    print(f"WARNING: ROR mismatch ({by_id[MERGE_FROM].ror_id} vs {by_id[MERGE_INTO].ror_id}) — "
          "expected for some merges, but double-check this is a true duplicate pair")

# The loser must be a pure aggregator shadow. A direct-ingest provenance means the loser is a
# real ingest target and this merge needs human review of the scraper/registry first.
AGGREGATOR_PROVENANCES = {
    "crossref_work", "crossref_work.grants", "crossref_work_funders",
    "datacite_work_funders", "europepmc_work_funders", "pubmed_work_funders",
}
prov = spark.sql(f"""
  SELECT provenance, COUNT(*) AS n FROM openalex.awards.openalex_awards_raw
  WHERE funder_id = {MERGE_FROM} GROUP BY provenance
""").collect()
print("loser awards_raw by provenance:", {r.provenance: r.n for r in prov})
direct = [r.provenance for r in prov if r.provenance not in AGGREGATOR_PROVENANCES]
assert not direct, f"loser has non-aggregator provenances {direct} — review before merging"

for tbl, col, val in [
    ("openalex.mid.work_funder", "funder_id", MERGE_FROM),
    ("openalex.awards.crossref_work_funders", "funder_id", MERGE_FROM),
    ("openalex.awards.datacite_work_funders", "funder_id", MERGE_FROM),
    ("openalex.awards.europepmc_work_funders", "funder_id", MERGE_FROM),
]:
    try:
        n = spark.sql(f"SELECT COUNT(*) AS n FROM {tbl} WHERE {col} = {val}").collect()[0].n
        print(f"{tbl}: {n} loser rows")
    except Exception as e:
        print(f"{tbl}: SKIP ({type(e).__name__})")

In [ ]:
# Schema: merge columns on mid.funder (+ common.funder mirror if present) ---
def ensure_merge_columns(table):
    cols = [f.name for f in spark.table(table).schema]
    todo = []
    if "merge_into_id" not in cols:
        todo.append("merge_into_id BIGINT")
    if "merge_into_date" not in cols:
        todo.append("merge_into_date TIMESTAMP")
    if todo:
        spark.sql(f"ALTER TABLE {table} ADD COLUMNS ({', '.join(todo)})")
        print(f"{table}: added {todo}")
    else:
        print(f"{table}: merge columns already present")

ensure_merge_columns("openalex.mid.funder")

# common.funder is a mirror maintained outside walden (only F4320* funders). Best effort:
# keep it consistent so DataCite DLT name/DOI matching resolves correctly; if its refresher
# rebuilds the table without these columns, the walden-side consumers stay defensive.
try:
    ensure_merge_columns("openalex.common.funder")
    HAVE_COMMON = True
except Exception as e:
    HAVE_COMMON = False
    print(f"common.funder: SKIP ({type(e).__name__}: {e})")

In [ ]:
# Winner absorbs loser names (display_name + alternate_titles) --------------
import json as _json

def _alts(raw):
    try:
        v = _json.loads(raw) if raw else []
        return v if isinstance(v, list) else []
    except Exception:
        return []

def absorb_names(table):
    r = spark.sql(f"""
      SELECT funder_id, display_name, alternate_titles FROM {table}
      WHERE funder_id IN ({MERGE_FROM}, {MERGE_INTO})
    """).collect()
    b = {x.funder_id: x for x in r}
    if MERGE_FROM not in b or MERGE_INTO not in b:
        print(f"{table}: one of the rows missing, skipping absorb")
        return
    merged = list(dict.fromkeys(
        _alts(b[MERGE_INTO].alternate_titles)
        + [b[MERGE_FROM].display_name]
        + _alts(b[MERGE_FROM].alternate_titles)
    ))
    merged = [n for n in merged if n and n != b[MERGE_INTO].display_name]
    lit = _json.dumps(merged, ensure_ascii=False).replace("\\", "\\\\").replace("'", "\\'")
    spark.sql(f"UPDATE {table} SET alternate_titles = '{lit}' WHERE funder_id = {MERGE_INTO}")
    print(f"{table}: winner alternate_titles -> {merged}")

absorb_names("openalex.mid.funder")
if HAVE_COMMON:
    absorb_names("openalex.common.funder")

In [ ]:
# Tombstone the loser -------------------------------------------------------
for table in ["openalex.mid.funder"] + (["openalex.common.funder"] if HAVE_COMMON else []):
    spark.sql(f"""
      UPDATE {table}
      SET merge_into_id = {MERGE_INTO}, merge_into_date = current_timestamp()
      WHERE funder_id = {MERGE_FROM}
    """)
    print(f"{table}: F{MERGE_FROM} tombstoned -> F{MERGE_INTO}")

In [ ]:
# One-time remap of stored work->funder edges -------------------------------
# The matcher notebooks re-derive these on their next run (and will resolve through
# merge_into_id after the branch lands); updating in place makes the merge take effect on
# the next downstream build instead of waiting a full rebuild cycle. Transient duplicate
# (work_id, funder_id) pairs are collapsed by every consumer (GROUP BY / dedup CTEs).
for tbl in [
    "openalex.mid.work_funder",
    "openalex.awards.crossref_work_funders",
    "openalex.awards.datacite_work_funders",
    "openalex.awards.europepmc_work_funders",
]:
    try:
        n = spark.sql(f"SELECT COUNT(*) AS n FROM {tbl} WHERE funder_id = {MERGE_FROM}").collect()[0].n
        if n:
            spark.sql(f"UPDATE {tbl} SET funder_id = {MERGE_INTO} WHERE funder_id = {MERGE_FROM}")
        print(f"{tbl}: remapped {n} rows")
    except Exception as e:
        print(f"{tbl}: SKIP ({type(e).__name__}: {e})")

# fulltext_work_funders stores the full OpenAlex URL string
try:
    ft_types = {f.name: f.dataType.simpleString()
                for f in spark.table("openalex.works.fulltext_work_funders").schema}
    winner_name = by_id[MERGE_INTO].display_name.replace("'", "\\'")
    if ft_types.get("funder_id") == "string":
        n = spark.sql(f"""
          SELECT COUNT(*) AS n FROM openalex.works.fulltext_work_funders
          WHERE funder_id = 'https://openalex.org/F{MERGE_FROM}'
        """).collect()[0].n
        if n:
            set_name = ", funder_display_name = '" + winner_name + "'" \
                if "funder_display_name" in ft_types else ""
            spark.sql(f"""
              UPDATE openalex.works.fulltext_work_funders
              SET funder_id = 'https://openalex.org/F{MERGE_INTO}'{set_name}
              WHERE funder_id = 'https://openalex.org/F{MERGE_FROM}'
            """)
    else:
        n = spark.sql(f"""
          SELECT COUNT(*) AS n FROM openalex.works.fulltext_work_funders
          WHERE funder_id = {MERGE_FROM}
        """).collect()[0].n
        if n:
            spark.sql(f"""
              UPDATE openalex.works.fulltext_work_funders
              SET funder_id = {MERGE_INTO} WHERE funder_id = {MERGE_FROM}
            """)
    print(f"fulltext_work_funders: remapped {n} rows")
except Exception as e:
    print(f"fulltext_work_funders: SKIP ({type(e).__name__}: {e})")

In [ ]:
# openalex_awards_raw: remap loser awards onto the winner -------------------
# Award identity is id = ABS(XXHASH64(CONCAT(funder_id, ':', lower(funder_award_id)))) % 9e9,
# identical across sources so duplicates collapse in CreateAwards. Remapping therefore MUST
# recompute the id under the winner; rows whose recomputed id already exists as a winner row
# in the same provenance are duplicates of awards the winner already has — delete those.
dup = spark.sql(f"""
  SELECT COUNT(*) AS n FROM openalex.awards.openalex_awards_raw t
  WHERE t.funder_id = {MERGE_FROM} AND EXISTS (
    SELECT 1 FROM openalex.awards.openalex_awards_raw w
    WHERE w.funder_id = {MERGE_INTO}
      AND w.provenance = t.provenance
      AND w.id = ABS(XXHASH64(CONCAT({MERGE_INTO}, ':', LOWER(t.funder_award_id)))) % 9000000000
  )
""").collect()[0].n
spark.sql(f"""
  DELETE FROM openalex.awards.openalex_awards_raw t
  WHERE t.funder_id = {MERGE_FROM} AND EXISTS (
    SELECT 1 FROM openalex.awards.openalex_awards_raw w
    WHERE w.funder_id = {MERGE_INTO}
      AND w.provenance = t.provenance
      AND w.id = ABS(XXHASH64(CONCAT({MERGE_INTO}, ':', LOWER(t.funder_award_id)))) % 9000000000
  )
""")
print(f"awards_raw: deleted {dup} loser rows duplicating existing winner awards")

w = by_id[MERGE_INTO]
dn = (w.display_name or "").replace("'", "\\'")
ror = f"'{w.ror_id}'" if w.ror_id else "CAST(NULL AS STRING)"
doi = f"'{w.doi}'" if w.doi else "CAST(NULL AS STRING)"
remap = spark.sql(f"""
  SELECT COUNT(*) AS n FROM openalex.awards.openalex_awards_raw WHERE funder_id = {MERGE_FROM}
""").collect()[0].n
spark.sql(f"""
  UPDATE openalex.awards.openalex_awards_raw
  SET id = ABS(XXHASH64(CONCAT({MERGE_INTO}, ':', LOWER(funder_award_id)))) % 9000000000,
      funder_id = {MERGE_INTO},
      funder = named_struct(
        'id', 'https://openalex.org/F{MERGE_INTO}',
        'display_name', '{dn}',
        'ror_id', {ror},
        'doi', {doi}
      ),
      works_api_url = CONCAT('https://api.openalex.org/works?filter=awards.id:G',
        ABS(XXHASH64(CONCAT({MERGE_INTO}, ':', LOWER(funder_award_id)))) % 9000000000),
      updated_date = current_timestamp()
  WHERE funder_id = {MERGE_FROM}
""")
print(f"awards_raw: remapped {remap} loser rows to F{MERGE_INTO} with recomputed ids")

In [ ]:
# Elasticsearch: redirect mapping + drop the stale loser doc ----------------
from elasticsearch import Elasticsearch

ELASTIC_URL = dbutils.secrets.get(scope="elastic", key="elastic_url")
FUNDERS_INDEX = "funders-v3"
MERGE_INDEX = "merge-funders"

es = Elasticsearch(hosts=[ELASTIC_URL], max_retries=3, request_timeout=180)

loser_url = f"https://openalex.org/F{MERGE_FROM}"
winner_url = f"https://openalex.org/F{MERGE_INTO}"

# mapping doc consumed by openalex-elastic-api funders_id_get (301 on miss)
es.index(index=MERGE_INDEX, id=loser_url,
         document={"id": loser_url, "merge_into_id": winner_url})
print(f"{MERGE_INDEX}: {loser_url} -> {winner_url}")

# the funders sync is a full-table upsert; after CreateFundersAPI filters the loser out,
# its doc would linger. Delete now (the sync's delete-stale block is the systemic backstop).
resp = es.options(ignore_status=[404]).delete(index=FUNDERS_INDEX, id=loser_url)
print(f"{FUNDERS_INDEX}: delete {loser_url} -> {resp.get('result', resp)}")
es.indices.refresh(index=FUNDERS_INDEX)

### Post-merge verification (run after the scheduled chain has cycled)

Databricks-side below; API-side once ES syncs:
- `GET /funders/F<winner>` — works/awards counts consolidated (sum minus overlap).
- `GET /funders/F<loser>` — **301** to the winner (after the elastic-api change deploys).
- Direct-ingest awards on the winner unchanged (Wellcome: 19,611 `wellcome_trust`).

In [ ]:
for label, sql in [
    ("mid.funder loser row (expect merge_into_id set)",
     f"SELECT funder_id, display_name, doi, merge_into_id FROM openalex.mid.funder WHERE funder_id = {MERGE_FROM}"),
    ("loser rows left in awards_raw (expect 0)",
     f"SELECT COUNT(*) AS n FROM openalex.awards.openalex_awards_raw WHERE funder_id = {MERGE_FROM}"),
    ("winner awards_raw by provenance",
     f"SELECT provenance, COUNT(*) AS n FROM openalex.awards.openalex_awards_raw WHERE funder_id = {MERGE_INTO} GROUP BY provenance ORDER BY n DESC"),
    ("loser edges left in junctions (expect 0s)",
     f"""SELECT
          (SELECT COUNT(*) FROM openalex.mid.work_funder WHERE funder_id = {MERGE_FROM}) AS work_funder,
          (SELECT COUNT(*) FROM openalex.awards.crossref_work_funders WHERE funder_id = {MERGE_FROM}) AS crossref,
          (SELECT COUNT(*) FROM openalex.awards.datacite_work_funders WHERE funder_id = {MERGE_FROM}) AS datacite,
          (SELECT COUNT(*) FROM openalex.awards.europepmc_work_funders WHERE funder_id = {MERGE_FROM}) AS europepmc"""),
]:
    print("==", label)
    spark.sql(sql).show(50, truncate=False)